# 🫁 Pneumonia Detection — DenseNet-121 + Grad-CAM
**Platform:** Kaggle (GPU T4 x2)  
**Model:** DenseNet-121 (pretrained on ImageNet)  
**Classes:** Normal | Bacterial Pneumonia | Viral Pneumonia  
**Priority:** High Recall / Sensitivity (minimize false negatives)  
**Output:** `densenet121_pneumonia.pth` + `class_weights.json`

---
> ⚠️ **Before running:** Go to **Settings → Accelerator → GPU T4 x2**

## 📦 Step 1 — Install & Import Libraries

In [ ]:
!pip install -q timm albumentations torchcam


In [ ]:
# Fix numpy-pandas binary incompatibility
!pip install --no-cache-dir --force-reinstall numpy==1.26.4 pandas==2.2.2


In [ ]:
# Install any missing libs (Kaggle usually has these)
!pip install -q timm albumentations torchcam

import os, json, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
import torchvision.transforms.functional as TF

import cv2
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📁 Step 2 — Dataset Path Setup

> 🔧 **Edit `DATA_ROOT` below** to match your Kaggle dataset path.  
> Usually: `/kaggle/input/<your-dataset-name>/`

In [ ]:

import os
print(os.listdir("/kaggle/input/datasets"))

In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/nisargads193"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/nisargads193/pneumonia"))

In [ ]:
# ============================================================
#  EDIT THIS: path to your uploaded dataset on Kaggle
# ============================================================
DATA_ROOT = Path('/kaggle/input/datasets/nisargads193/pneumonia') # <-- change this

# Expected folder structure:
# DATA_ROOT/
#   train/
#     NORMAL/
#     BACTERIA/   (or BACTERIAL)
#     VIRUS/      (or VIRAL)
#   val/
#     ...
#   test/
#     ...

TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

# Class mapping (update folder names if yours differ)
CLASS_MAP = {
    'NORMAL':   0,
    'BACTERIA': 1,
    'VIRAL':    2,
}
CLASS_NAMES = ['Normal', 'Bacterial Pneumonia', 'Viral Pneumonia']

# Verify paths exist
for split, path in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    exists = path.exists()
    print(f'  {split}: {path}  [{"OK" if exists else "NOT FOUND - check DATA_ROOT"}]')
    if exists:
        for cls_folder in path.iterdir():
            if cls_folder.is_dir():
                n = len(list(cls_folder.glob('*.*')))
                print(f'    └─ {cls_folder.name}: {n} images')

## 🔄 Step 3 — Data Augmentation & Transforms

Heavy augmentation on training set. Chest X-rays need careful augmentation — no color jitter, only geometry and contrast.

In [ ]:
IMG_SIZE = 224  # DenseNet-121 standard input

# Mean/std from ImageNet (since we use pretrained DenseNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # OK for X-rays
    transforms.RandomAutocontrast(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms ready.')

## 🗂️ Step 4 — Custom Dataset Class

In [ ]:
class ChestXRayDataset(Dataset):
    """
    Loads chest X-ray images from folder structure:
      root/
        NORMAL/img1.jpeg ...
        BACTERIA/img1.jpeg ...
        VIRAL/img1.jpeg ...
    """
    def __init__(self, root_dir, class_map, transform=None):
        self.root_dir   = Path(root_dir)
        self.class_map  = class_map
        self.transform  = transform
        self.samples    = []  # (path, label)
        self._load_samples()

    def _load_samples(self):
        exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        for folder in self.root_dir.iterdir():
            folder_name = folder.name.upper()
            # Flexible matching: BACTERIA matches BACTERIA/BACTERIAL, VIRUS matches VIRUS/VIRAL
            label = None
            for key, idx in self.class_map.items():
                if key in folder_name or folder_name in key:
                    label = idx
                    break
            if label is None:
                print(f'  ⚠️  Skipping unrecognized folder: {folder.name}')
                continue
            for img_path in folder.iterdir():
                if img_path.suffix.lower() in exts:
                    self.samples.append((str(img_path), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_counts(self):
        labels = [s[1] for s in self.samples]
        return Counter(labels)

# Build datasets
train_dataset = ChestXRayDataset(TRAIN_DIR, CLASS_MAP, transform=train_transforms)
test_dataset  = ChestXRayDataset(TEST_DIR,  CLASS_MAP, transform=val_test_transforms)

print(f'Train: {len(train_dataset)} images')
print(f'Test:  {len(test_dataset)} images')

counts = train_dataset.get_class_counts()
print('\nTrain class distribution:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {counts[i]}')

## ⚖️ Step 5 — Class Imbalance Handling

Two strategies combined:
1. **Weighted Random Sampler** — over-samples minority classes during training
2. **Weighted Cross-Entropy Loss** — penalizes wrong predictions on rare classes more

In [ ]:
# Step 5 — Class Imbalance Handling

from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

# -----------------------------
# Get labels from training data
# -----------------------------
train_labels = [s[1] for s in train_dataset.samples]

# Count samples per class
class_counts = Counter(train_labels)

print("Class distribution in training set:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name}: {class_counts[i]} images")

# -----------------------------
# Weighted Random Sampler
# -----------------------------
class_weights = {cls: 1.0 / cnt for cls, cnt in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# -----------------------------
# DataLoaders
# -----------------------------
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# -----------------------------
# Weighted Loss Function
# -----------------------------
total = sum(class_counts.values())

loss_weights = torch.tensor([
    total / (len(class_counts) * class_counts[i])
    for i in range(len(CLASS_NAMES))
], dtype=torch.float).to(device)

# Increase penalty for missing pneumonia cases
loss_weights[1] *= 1.5   # Bacterial Pneumonia
loss_weights[2] *= 1.5   # Viral Pneumonia

criterion = torch.nn.CrossEntropyLoss(weight=loss_weights)

print("\nLoss weights (higher = more penalty if misclassified):")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name}: {loss_weights[i].item():.4f}")

## 🏗️ Step 6 — DenseNet-121 Model Setup

Using pretrained DenseNet-121. We replace the final classifier layer with our 3-class head. The last `denseblock` layers are unfrozen for fine-tuning.

In [ ]:
def build_densenet121(num_classes=3, dropout_p=0.4):
    """
    Pretrained DenseNet-121 with custom classifier head.
    Strategy: Freeze early layers, fine-tune denseblock3+4 + classifier.
    """
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

    # --- Freeze all layers first ---
    for param in model.parameters():
        param.requires_grad = False

    # --- Unfreeze denseblock3, denseblock4, norm5 (top feature extractor layers) ---
    for name, param in model.features.named_parameters():
        if any(x in name for x in ['denseblock3', 'denseblock4', 'norm5', 'transition3']):
            param.requires_grad = True

    # --- Replace classifier ---
    in_features = model.classifier.in_features  # 1024 for DenseNet-121
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=dropout_p),
        nn.Linear(512, num_classes)
    )

    return model


model = build_densenet121(num_classes=3)
model = model.to(device)

# Use DataParallel if 2 GPUs available (Kaggle T4 x2)
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    model = nn.DataParallel(model)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)')

## ⚙️ Step 7 — Optimizer, Scheduler & Config

In [ ]:
# ============================================================
#  TRAINING CONFIG — tune these
# ============================================================
NUM_EPOCHS    = 30       # 25-35 is enough for fine-tuning
LR_HEAD       = 1e-3     # Higher LR for new classifier head
LR_BACKBONE   = 1e-4     # Lower LR for pretrained backbone layers
WEIGHT_DECAY  = 1e-4
PATIENCE      = 7        # Early stopping patience
RECALL_THRESHOLD = 0.90  # Target recall for pneumonia classes

# Separate param groups for different learning rates
m = model.module if isinstance(model, nn.DataParallel) else model

optimizer = optim.AdamW([
    {'params': m.classifier.parameters(),  'lr': LR_HEAD},
    {'params': m.features.parameters(),    'lr': LR_BACKBONE},
], weight_decay=WEIGHT_DECAY)

# CosineAnnealingLR: smoothly decreases LR — great for fine-tuning
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

print(f'Optimizer: AdamW | Head LR: {LR_HEAD} | Backbone LR: {LR_BACKBONE}')
print(f'Scheduler: CosineAnnealing | Epochs: {NUM_EPOCHS}')
print(f'Early stopping patience: {PATIENCE} epochs')

## 🏋️ Step 8 — Training Loop with High-Recall Tracking

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

def compute_metrics(all_labels, all_preds):
    """Returns per-class recall and macro recall."""
    recalls = recall_score(all_labels, all_preds, average=None, zero_division=0)
    macro_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return recalls, macro_recall


def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    is_training = (phase == 'train')
    model.train() if is_training else model.eval()

    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if is_training:
                optimizer.zero_grad()

            outputs = model(images)
            loss    = criterion(outputs, labels)

            if is_training:
                loss.backward()
                # Gradient clipping — prevents exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    recalls, macro_recall = compute_metrics(all_labels, all_preds)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return epoch_loss, acc, recalls, macro_recall, all_labels, all_preds

print('Training utilities ready.')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
           'val_recall_normal': [], 'val_recall_bacteria': [], 'val_recall_virus': [],
           'val_macro_recall': []}

best_val_loss   = float('inf')
best_macro_recall = 0.0
patience_counter = 0
best_model_wts  = copy.deepcopy(model.state_dict())

print(f'Starting training for {NUM_EPOCHS} epochs...\n')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Val Loss":>8} | {"Val Acc":>7} | '
      f'{"Recall-N":>8} | {"Recall-B":>8} | {"Recall-V":>8} | {"Macro-R":>7}')
print('-' * 90)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc, tr_rec, tr_mrec, _, _ = run_epoch(
        model, train_loader, criterion, optimizer, phase='train')

    vl_loss, vl_acc, vl_rec, vl_mrec, vl_labels, vl_preds = run_epoch(
        model, test_loader, criterion, phase='test')

    scheduler.step()

    # Log history
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_recall_normal'].append(vl_rec[0])
    history['val_recall_bacteria'].append(vl_rec[1])
    history['val_recall_virus'].append(vl_rec[2])
    history['val_macro_recall'].append(vl_mrec)

    elapsed = time.time() - t0
    print(f'{epoch:>5} | {tr_loss:>10.4f} | {vl_loss:>8.4f} | {vl_acc:>7.3f} | '
          f'{vl_rec[0]:>8.3f} | {vl_rec[1]:>8.3f} | {vl_rec[2]:>8.3f} | {vl_mrec:>7.3f} '
          f'({elapsed:.0f}s)')

    # --- Save best model: prioritize macro recall (high sensitivity) ---
    # A model is "best" if its macro recall improves OR if recall is tied and loss is better
    is_best = (vl_mrec > best_macro_recall) or \
              (abs(vl_mrec - best_macro_recall) < 0.001 and vl_loss < best_val_loss)

    if is_best:
        best_val_loss = vl_loss
        best_macro_recall = vl_mrec
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, OUTPUT_DIR / 'densenet121_pneumonia_best.pth')
        patience_counter = 0
        print(f'         ✅ New best model saved (macro recall={vl_mrec:.3f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\n⏹ Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

# Restore best weights
model.load_state_dict(best_model_wts)
print(f'\n✅ Training complete. Best macro recall: {best_macro_recall:.4f}')

## 📊 Step 9 — Training Curves

In [ ]:
epochs_ran = len(history['train_loss'])
ep = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DenseNet-121 Training — Pneumonia Detection', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(ep, history['train_loss'], label='Train', color='royalblue')
axes[0].plot(ep, history['val_loss'],   label='Val',   color='tomato')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(ep, history['train_acc'], label='Train', color='royalblue')
axes[1].plot(ep, history['val_acc'],   label='Val',   color='tomato')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Per-class Recall (most important chart)
axes[2].plot(ep, history['val_recall_normal'],   label='Normal',    color='mediumseagreen')
axes[2].plot(ep, history['val_recall_bacteria'], label='Bacterial', color='darkorange')
axes[2].plot(ep, history['val_recall_virus'],    label='Viral',     color='mediumpurple')
axes[2].plot(ep, history['val_macro_recall'],    label='Macro',     color='black', linestyle='--', linewidth=2)
axes[2].axhline(y=RECALL_THRESHOLD, color='red', linestyle=':', label=f'Target ({RECALL_THRESHOLD})')
axes[2].set_title('Validation Recall (Sensitivity)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 🧪 Step 10 — Test Set Evaluation

In [ ]:
ts_loss, ts_acc, ts_rec, ts_mrec, ts_labels, ts_preds = run_epoch(
    model, test_loader, criterion, phase='val')  # no optimizer = eval mode

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(f'Accuracy:      {ts_acc:.4f}')
print(f'Macro Recall:  {ts_mrec:.4f}')
print()
print('Per-class Recall (Sensitivity):')
for i, name in enumerate(CLASS_NAMES):
    status = '✅' if ts_rec[i] >= RECALL_THRESHOLD else '⚠️ '
    print(f'  {status} {name}: {ts_rec[i]:.4f}')
print()
print('Classification Report:')
print(classification_report(ts_labels, ts_preds, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(ts_labels, ts_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(['Normal', 'Bacterial', 'Viral'], rotation=30)
ax.set_yticklabels(['Normal', 'Bacterial', 'Viral'])
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()*0.5 else 'black', fontsize=14)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 🗺️ Step 11 — Grad-CAM Heatmap Generation

Grad-CAM uses gradients of the predicted class flowing back to the last convolutional layer (`denseblock4`) to create a heatmap showing **where** the model is looking.

In [ ]:
class GradCAM:
    """
    Grad-CAM for DenseNet-121.
    Target layer: features.denseblock4 (last dense block before classifier).
    """
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        # Unwrap DataParallel if needed
        m = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        target_layer = m.features.denseblock4

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        """
        Args:
            input_tensor: [1, 3, H, W] tensor
            class_idx: int or None (uses predicted class if None)
        Returns:
            cam: numpy array [H, W] with values in [0, 1]
            pred_class: predicted class index
            probs: softmax probabilities
        """
        self.model.eval()
        input_tensor = input_tensor.to(device).requires_grad_(True)

        output = self.model(input_tensor)
        probs  = torch.softmax(output, dim=1).squeeze().detach().cpu().numpy()
        pred_class = int(output.argmax(dim=1).item())

        if class_idx is None:
            class_idx = pred_class

        # Backprop for target class
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1.0
        output.backward(gradient=one_hot)

        # Global Average Pooling over gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # [1, C, 1, 1]
        cam = (weights * self.activations).sum(dim=1).squeeze()   # [H, W]

        # ReLU + Normalize
        cam = torch.relu(cam).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, pred_class, probs


def overlay_heatmap(original_pil, cam, alpha=0.45, colormap=cv2.COLORMAP_JET):
    """
    Overlays Grad-CAM heatmap on original image.
    Returns: PIL Image with red heatmap overlay.
    """
    img_np = np.array(original_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.uint8(alpha * heatmap + (1 - alpha) * img_np)
    return Image.fromarray(overlay)


gradcam = GradCAM(model)
print('Grad-CAM ready.')

In [ ]:
for name, module in model.named_modules():
    print(name)

In [ ]:
# ==============================
# Grad-CAM Pneumonia Visualization (ONE CELL)
# ==============================

!pip install -q grad-cam

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Prepare model
model = model.to(device)
model.eval()

# Handle DataParallel
cam_model = model.module if hasattr(model, "module") else model

# DenseNet last feature block
target_layers = [cam_model.features.denseblock4]

# Create GradCAM
cam = GradCAM(
    model=cam_model,
    target_layers=target_layers
)

# GradCAM function
def generate_gradcam(image):

    rgb_img = np.array(image.resize((224,224))) / 255.0

    input_tensor = val_test_transforms(image).unsqueeze(0).to(device)

    outputs = cam_model(input_tensor)

    pred_class = torch.argmax(outputs).item()

    targets = [ClassifierOutputTarget(pred_class)]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    cam_image = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )

    probs = torch.softmax(outputs, dim=1)[0].detach().cpu().numpy()

    return cam_image, pred_class, probs


# Collect sample images
samples_per_class = {i: [] for i in range(len(CLASS_NAMES))}

for img_path, label in test_dataset.samples:
    if len(samples_per_class[label]) < 3:
        samples_per_class[label].append(img_path)


# Plot GradCAM
rows = len(CLASS_NAMES)
cols = 6

fig, axes = plt.subplots(rows, cols, figsize=(18,8))

for cls_idx, class_name in enumerate(CLASS_NAMES):

    for i, img_path in enumerate(samples_per_class[cls_idx]):

        image = Image.open(img_path).convert("RGB")

        cam_img, pred_class, probs = generate_gradcam(image)

        # Original
        axes[cls_idx][i*2].imshow(image.resize((224,224)))
        axes[cls_idx][i*2].set_title(f"True: {class_name}")
        axes[cls_idx][i*2].axis("off")

        # GradCAM
        pred_name = CLASS_NAMES[pred_class]
        conf = probs[pred_class]

        color = "green" if pred_class == cls_idx else "red"

        axes[cls_idx][i*2+1].imshow(cam_img)
        axes[cls_idx][i*2+1].set_title(
            f"Pred: {pred_name}\nConf: {conf:.2%}",
            color=color
        )
        axes[cls_idx][i*2+1].axis("off")

plt.tight_layout()
plt.show()

## 💾 Step 12 — Save Everything for Local Deployment

In [ ]:
# 1. Save final model weights
m = model.module if isinstance(model, nn.DataParallel) else model
torch.save(m.state_dict(), OUTPUT_DIR / 'densenet121_pneumonia.pth')
print('Saved: densenet121_pneumonia.pth')

# 2. Save class metadata
metadata = {
    'class_names':  CLASS_NAMES,
    'class_map':    CLASS_MAP,
    'img_size':     IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std':  IMAGENET_STD,
    'model_arch':   'densenet121',
    'num_classes':  3,
    'test_accuracy': float(ts_acc),
    'test_macro_recall': float(ts_mrec),
    'test_recall_per_class': {name: float(ts_rec[i]) for i, name in enumerate(CLASS_NAMES)},
}
with open(OUTPUT_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved: model_metadata.json')

# 3. List all output files
print('\n📁 Output files in /kaggle/working:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1e6
    print(f'  {f.name:40s}  {size:.1f} MB')

## ✅ Step 13 — How to Use This in Your Local Project

After downloading `densenet121_pneumonia.pth` and `model_metadata.json`, use this snippet in your local Python project:

```python
import torch
import json
from torchvision import models, transforms
from PIL import Image

# Load metadata
with open('model_metadata.json') as f:
    meta = json.load(f)

# Rebuild model architecture
model = models.densenet121(weights=None)
import torch.nn as nn
model.classifier = nn.Sequential(
    nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, 3)
)
model.load_state_dict(torch.load('densenet121_pneumonia.pth', map_location='cpu'))
model.eval()

# Preprocess and predict
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(meta['imagenet_mean'], meta['imagenet_std']),
])
img = Image.open('xray.jpg').convert('RGB')
tensor = transform(img).unsqueeze(0)
with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze().numpy()
pred_class = probs.argmax()
print(f'Prediction: {meta["class_names"][pred_class]} ({probs[pred_class]:.2%})')
```
